In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")
 

In [2]:
df = pd.read_csv("C:\\Users\\AIA\\Downloads/teamify_login_logs_final.csv")

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%Y %I:%M:%S %p")

# 1.2 – Normalize text columns
for col in ["location", "device", "browser", "login_status", "attack_type", "IP_Address"]:
    df[col] = df[col].astype(str).str.strip().str.lower()
 
# 1.3 – Duplicates
before = len(df)
df.drop_duplicates(inplace=True)
 
# 1.4 – Missing values check
missing = df.isnull().sum().sum()

# 1.5 – Sort by user and time
df.sort_values(["user_id", "timestamp"], inplace=True)
df.reset_index(drop=True, inplace=True)

# print(f"\n Dataset Summary:")
# print(f"   Total records  : {len(df):,}")
# print(f"   Unique users   : {df['user_id'].nunique():,}")
# print(f"   Date range     : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
# print(f"   Failed logins  : {(df['login_status'] == 'failed').sum():,}")
# print(f"   Labeled attacks: {(df['attack_type'] != 'none').sum():,}")
 

In [5]:
# 2.1 – Time-based features
df["login_hour"]    = df["timestamp"].dt.hour
df["day_of_week"]   = df["timestamp"].dt.dayofweek   # 0=Monday, 6=Sunday
df["is_weekend"]    = df["day_of_week"].isin([5, 6]).astype(int)
df["login_minute"]  = df["timestamp"].dt.minute
 
# 2.2 – Time since last login (per user)
df["time_since_last_login"] = (
    df.groupby("user_id")["timestamp"]
    .diff()
    .dt.total_seconds()
    .fillna(0)
)
 
# 2.3 – IP change flag (per user)
df["prev_ip"] = df.groupby("user_id")["IP_Address"].shift(1)
df["ip_changed"] = (df["IP_Address"] != df["prev_ip"]).astype(int)
df.loc[df["time_since_last_login"] == 0, "ip_changed"] = 0   # first login – no change
 
# 2.4 – Location change flag (per user)
df["prev_location"] = df.groupby("user_id")["location"].shift(1)
df["location_changed"] = (df["location"] != df["prev_location"]).astype(int)
df.loc[df["time_since_last_login"] == 0, "location_changed"] = 0
 
# 2.5 – Device change flag (per user)
df["prev_device"] = df.groupby("user_id")["device"].shift(1)
df["device_changed"] = (df["device"] != df["prev_device"]).astype(int)
df.loc[df["time_since_last_login"] == 0, "device_changed"] = 0
 
# 2.6 – Rolling failed attempts (last 5 logins per user)
df["is_failed"] = (df["login_status"] == "failed").astype(int)
df["rolling_failed_5"] = (
    df.groupby("user_id")["is_failed"]
    .transform(lambda x: x.rolling(5, min_periods=1).sum())
)
 
# 2.7 – Per-user behavioral statistics (computed on full history)
user_stats = df.groupby("user_id").agg(
    user_avg_hour        = ("login_hour", "mean"),
    user_std_hour        = ("login_hour", "std"),
    user_avg_gap         = ("time_since_last_login", "mean"),
    user_std_gap         = ("time_since_last_login", "std"),
    user_total_logins    = ("login_hour", "count"),
    user_avg_failed      = ("is_failed", "mean"),
    user_unique_ips      = ("IP_Address", "nunique"),
    user_unique_locs     = ("location", "nunique"),
    user_unique_devices  = ("device", "nunique"),
).reset_index()
 
user_stats["user_std_hour"] = user_stats["user_std_hour"].fillna(0)
user_stats["user_std_gap"]  = user_stats["user_std_gap"].fillna(0)
 
df = df.merge(user_stats, on="user_id", how="left")
 
# 2.8 – Deviation features: how much does THIS login deviate from user's own pattern?
df["hour_deviation"] = (df["login_hour"] - df["user_avg_hour"]).abs()
 
# Safe z-score: deviation normalized by user's own std
df["hour_zscore"] = np.where(
    df["user_std_hour"] > 0,
    (df["login_hour"] - df["user_avg_hour"]) / df["user_std_hour"],
    0
)
df["gap_zscore"] = np.where(
    df["user_std_gap"] > 0,
    (df["time_since_last_login"] - df["user_avg_gap"]) / df["user_std_gap"],
    0
)
 
# 2.9 – Encode categorical features
le_device  = LabelEncoder()
le_browser = LabelEncoder()
df["device_enc"]  = le_device.fit_transform(df["device"])
df["browser_enc"] = le_browser.fit_transform(df["browser"])
 
print(" Features created:")
feature_list = [
    "login_hour", "day_of_week", "is_weekend",
    "time_since_last_login", "ip_changed", "location_changed", "device_changed",
    "rolling_failed_5", "hour_deviation", "hour_zscore", "gap_zscore",
    "device_enc", "browser_enc",
    "user_avg_hour", "user_std_hour", "user_avg_gap", "user_std_gap",
    "user_avg_failed", "user_unique_ips", "user_unique_locs", "user_unique_devices"
]
for f in feature_list:
    print(f"   • {f}")
 
 

 Features created:
   • login_hour
   • day_of_week
   • is_weekend
   • time_since_last_login
   • ip_changed
   • location_changed
   • device_changed
   • rolling_failed_5
   • hour_deviation
   • hour_zscore
   • gap_zscore
   • device_enc
   • browser_enc
   • user_avg_hour
   • user_std_hour
   • user_avg_gap
   • user_std_gap
   • user_avg_failed
   • user_unique_ips
   • user_unique_locs
   • user_unique_devices


In [6]:
FEATURES = [
    "login_hour", "day_of_week", "is_weekend",
    "time_since_last_login", "ip_changed", "location_changed", "device_changed",
    "rolling_failed_5", "hour_deviation", "hour_zscore", "gap_zscore",
    "device_enc", "browser_enc",
    "user_avg_failed", "user_unique_ips", "user_unique_locs"
]
 
# We train one Isolation Forest per user if they have enough records
# For users with very few records, we use a global fallback model.
 
MIN_RECORDS_PER_USER = 5
CONTAMINATION = 0.05   # expected ~5% anomaly rate
 
anomaly_scores  = np.zeros(len(df))
anomaly_labels  = np.zeros(len(df), dtype=int)   # 1 = anomaly, 0 = normal
model_used      = np.full(len(df), "global", dtype=object)
 
# Train global fallback model first
global_model = IsolationForest(
    n_estimators=100,
    contamination=CONTAMINATION,
    random_state=42
)
global_model.fit(df[FEATURES].fillna(0))
 
users = df["user_id"].unique()
trained_per_user = 0
 
for uid in users:
    mask = df["user_id"] == uid
    user_df = df[mask]
    X = user_df[FEATURES].fillna(0).values
 
    if len(user_df) >= MIN_RECORDS_PER_USER:
        model = IsolationForest(
            n_estimators=100,
            contamination=CONTAMINATION,
            random_state=42
        )
        model.fit(X)
        scores = model.decision_function(X)    # lower = more anomalous
        preds  = model.predict(X)              # -1 = anomaly, 1 = normal
        model_used[mask] = "per_user"
        trained_per_user += 1
    else:
        scores = global_model.decision_function(X)
        preds  = global_model.predict(X)
        model_used[mask] = "global_fallback"
 
    anomaly_scores[mask] = scores
    anomaly_labels[mask] = (preds == -1).astype(int)
 
df["anomaly_score"]  = anomaly_scores
df["is_anomaly"]     = anomaly_labels
df["model_used"]     = model_used
 
print(f" Per-user models trained : {trained_per_user:,} users")
print(f" Global fallback used    : {(df['model_used']=='global_fallback').sum():,} records")
print(f"\n Detection Results:")
print(f"   Normal logins  : {(df['is_anomaly']==0).sum():,} ({(df['is_anomaly']==0).mean()*100:.1f}%)")
print(f"   Anomalies flagged: {(df['is_anomaly']==1).sum():,} ({(df['is_anomaly']==1).mean()*100:.1f}%)")
 

 Per-user models trained : 3,000 users
 Global fallback used    : 0 records

 Detection Results:
   Normal logins  : 55,924 (93.2%)
   Anomalies flagged: 4,076 (6.8%)


In [7]:
# Ground truth: attack_type != 'none'
df["is_true_anomaly"] = (df["attack_type"] != "none").astype(int)
 
TP = ((df["is_anomaly"] == 1) & (df["is_true_anomaly"] == 1)).sum()
FP = ((df["is_anomaly"] == 1) & (df["is_true_anomaly"] == 0)).sum()
FN = ((df["is_anomaly"] == 0) & (df["is_true_anomaly"] == 1)).sum()
TN = ((df["is_anomaly"] == 0) & (df["is_true_anomaly"] == 0)).sum()
 
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
 
print(f"\n Confusion Matrix (vs. labeled attack_type):")
print(f"   True Positives  (TP): {TP:,}  ← correctly caught attacks")
print(f"   False Positives (FP): {FP:,}  ← normal flagged as anomaly")
print(f"   False Negatives (FN): {FN:,}  ← attacks missed")
print(f"   True Negatives  (TN): {TN:,}  ← correctly marked normal")
print(f"\n Metrics:")
print(f"   Precision : {precision:.3f}")
print(f"   Recall    : {recall:.3f}")
print(f"   F1 Score  : {f1:.3f}")
 
print(f"\n Anomaly breakdown by attack type:")
print(df[df["is_anomaly"]==1]["attack_type"].value_counts().to_string())
 


 Confusion Matrix (vs. labeled attack_type):
   True Positives  (TP): 1,768  ← correctly caught attacks
   False Positives (FP): 2,308  ← normal flagged as anomaly
   False Negatives (FN): 1,232  ← attacks missed
   True Negatives  (TN): 54,692  ← correctly marked normal

 Metrics:
   Precision : 0.434
   Recall    : 0.589
   F1 Score  : 0.500

 Anomaly breakdown by attack type:
attack_type
none            2308
brute_force      801
geo_anomaly      678
burst_logins     289


In [8]:
display_cols = [
    "user_id", "timestamp", "location", "device", "browser",
    "login_status", "failed_attempts", "attack_type",
    "login_hour", "ip_changed", "location_changed",
    "hour_zscore", "rolling_failed_5", "anomaly_score", "is_anomaly"
]
 
print("\n NORMAL Logins (sample of 5):")
normal_sample = df[df["is_anomaly"] == 0].sample(5, random_state=42)[display_cols]
print(normal_sample.to_string(index=False))
 
print("\n  ANOMALOUS Logins (sample of 10):")
anomaly_sample = (
    df[df["is_anomaly"] == 1]
    .sort_values("anomaly_score")  # most anomalous first
    .head(10)[display_cols]
)
print(anomaly_sample.to_string(index=False))
 
print("\n  CAUGHT ATTACKS by type (sample):")
for atype in ["brute_force", "geo_anomaly", "burst_logins"]:
    subset = df[(df["attack_type"] == atype) & (df["is_anomaly"] == 1)]
    if not subset.empty:
        row = subset.iloc[0][display_cols]
        print(f"\n  [{atype.upper()}]")
        for col in display_cols:
            print(f"     {col:25s}: {row[col]}")
 


 NORMAL Logins (sample of 5):
 user_id           timestamp     location  device browser login_status  failed_attempts attack_type  login_hour  ip_changed  location_changed  hour_zscore  rolling_failed_5  anomaly_score  is_anomaly
    2031 2024-10-15 13:06:30   beirut, lb     ios  chrome      success                0        none          13           1                 0    -0.188127               0.0       0.124631           0
     903 2024-01-17 17:27:30    amman, jo android firefox      success                0        none          17           1                 0     1.308760               0.0       0.113322           0
    2789 2024-04-19 11:42:17    paris, fr windows  safari      success                0        none          11           1                 0     1.091820               1.0       0.015688           0
    1496 2024-03-10 12:23:03 new york, us     ios  chrome      success                0        none          12           1                 0     0.948683               

In [9]:
out_cols = [
    "user_id", "timestamp", "location", "device", "browser",
    "login_status", "failed_attempts", "attack_type", "IP_Address",
    "login_hour", "day_of_week", "is_weekend",
    "time_since_last_login", "ip_changed", "location_changed", "device_changed",
    "rolling_failed_5", "hour_deviation", "hour_zscore", "gap_zscore",
    "user_avg_hour", "user_std_hour", "user_avg_failed",
    "user_unique_ips", "user_unique_locs", "user_unique_devices",
    "anomaly_score", "is_anomaly", "model_used", "is_true_anomaly"
]
 
df[out_cols].to_csv(
    'C:/Users/AIA/Downloads/teamify_anomaly_results.csv',
    index=False
), 
print("\n\n" + "=" * 60)
print(" OUTPUT SAVED → teamify_anomaly_results.csv")
print("=" * 60)
print(f"   Rows  : {len(df):,}")
print(f"   Cols  : {len(out_cols)}")
print("   Pipeline: Preprocessing → Feature Engineering → Training → Prediction ")
 



 OUTPUT SAVED → teamify_anomaly_results.csv
   Rows  : 60,000
   Cols  : 30
   Pipeline: Preprocessing → Feature Engineering → Training → Prediction 
